# Dataset: ClueWeb09 Category B (TREC Web Track 2009-2012, diversity task)

- [clueweb09/catb](https://ir-datasets.com/clueweb09.html#clueweb09/catb)
- [clueweb09/catb/trec-web-2009/diversity](https://ir-datasets.com/clueweb09.html#clueweb09/catb/trec-web-2009/diversity) (and 2010/2011/2012)

## Install python modules

In [ ]:
import sys
!{sys.executable} -m pip install ir_datasets pandas ir_measures

## Obtain the corpus

ClueWeb09 is a licensed corpus: obtain it from CMU and place it under
`~/.ir_datasets/clueweb09/corpus` (see the ir_datasets page for details).
Queries and qrels download automatically.

In [ ]:
import ir_datasets
dataset_name = "clueweb09/catb"
dataset = ir_datasets.load(dataset_name)

Corpus size

In [ ]:
print(dataset.docs_count())

Document fields and sample data

Documents are raw `WarcDoc`s (bytes HTML); `default_text()` performs
HTML-to-text extraction. The extracted title arrives on the first line
(pages yielding a single line have no separate title).

In [ ]:
print(dataset.docs_cls().__annotations__)

In [ ]:
doc = next(dataset.docs_iter())
print(doc.doc_id, "|", doc.url)
print(doc.default_text()[:300])

## Obtain the queries/qrels

The **diversity task** variants of the TREC Web Track (50 topics per year).
Topics carry subtopics (faceted/ambiguous); qrels are judged per subtopic.

In [ ]:
years = ["2009", "2010", "2011", "2012"]
diversity = {y: ir_datasets.load(f"clueweb09/catb/trec-web-{y}/diversity") for y in years}
for y, d in diversity.items():
    print(y, "queries:", d.queries_count(), "qrels:", d.qrels_count())

Query fields and a sample topic with its subtopics

In [ ]:
print(diversity['2009'].queries_cls().__annotations__)

In [ ]:
import pandas as pd
topic = next(diversity["2009"].queries_iter())
print(topic.query_id, "|", topic.query, "|", topic.type)
pd.DataFrame(topic.subtopics)

Show queries

In [ ]:
pd.DataFrame([(q.query_id, q.query, q.type, len(q.subtopics)) for q in diversity["2009"].queries_iter()],
             columns=["query_id", "query", "type", "n_subtopics"])

Qrel fields and data

Qrels carry the subtopic id; relevance grades include **-2 (spam)**.

In [ ]:
print(diversity['2009'].qrels_cls().__annotations__)

In [ ]:
pd.DataFrame(diversity["2009"].qrels_iter())

Relevance grade distribution per year

In [ ]:
for y, d in diversity.items():
    df = pd.DataFrame(d.qrels_iter())
    print(y, dict(df["relevance"].value_counts().sort_index()))

## Evaluation convention

Evaluation is handled by [ir_measures](https://ir-measur.es/); its `ndeval`
provider supplies the track's diversity measures.

In [ ]:
import ir_measures
from ir_measures import parse_measure
measures = [parse_measure("alpha_nDCG@10"), parse_measure("ERR_IA@10"), parse_measure("NRBP")]
print(measures)
# usage once a run exists:
# ir_measures.calc_aggregate(measures, diversity["2009"].qrels_iter(), run)